In [ ]:
import pandas as pd

# Load the DataFrame from the relative path
df_midlands = pd.read_csv("../dataset/midlands_subset.csv")

# Display the first few rows to verify
df_midlands.head()

In [ ]:
#1. Explore the data ETL
#Display field, check for missing rows,
print(df_midlands.dtypes)
df_midlands.describe(include='all')


In [ ]:
## Section 1 - Exploring features that relate to energy.
## We're looking only at houses and the ones given a lower rating E to G.   
##  We'll see what causes the low rating very quick by the Percentage_Poor_or_Very_Poor

# 1. Filter for low-rated properties (E, F, and G)
low_rated_mask = df_midlands['CURRENT_ENERGY_RATING'].isin(['E', 'F', 'G'])
df_low = df_midlands[low_rated_mask]

# 2. Define the structural rating columns we want to test
components = [
    'WALLS_ENERGY_EFF',
    'ROOF_ENERGY_EFF',
    'WINDOWS_ENERGY_EFF',
    'MAINHEAT_ENERGY_EFF'
]

# 3. Calculate percentage of 'Very Poor' or 'Poor' ratings per component
results = {}
total_low_rated = len(df_low)

for col in components:
    # Count properties with Poor or Very Poor ratings
    poor_count = df_low[col].isin(['Poor', 'Very Poor']).sum()
    
    # Calculate percentage
    percentage = (poor_count / total_low_rated) * 100
    results[col] = round(percentage, 2)

# Display summary dataframe
df_study = pd.DataFrame(
    list(results.items()), 
    columns=['Component', 'Percentage_Poor_or_Very_Poor']
).sort_values(by='Percentage_Poor_or_Very_Poor', ascending=False)

print(df_study)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Define components to analyze
components = [
    'WALLS_ENERGY_EFF',
    'ROOF_ENERGY_EFF',
    'WINDOWS_ENERGY_EFF',
    'MAINHEAT_ENERGY_EFF'
]

# 2. Count 'Poor' or 'Very Poor' properties per component in df_low
poor_counts = {}
for col in components:
    count = df_low[col].isin(['Poor', 'Very Poor']).sum()
    poor_counts[col] = count

# Convert to a Series for plotting
poor_series = pd.Series(poor_counts)

# 3. Draw a pie chart to see how each component contributes to the poor-rating pool
plt.figure(figsize=(8, 8))
plt.pie(
    poor_series,
    labels=poor_series.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=['#ff9999','#66b3ff','#99ff99','#ffcc99'],
    explode=(0.05, 0.05, 0.05, 0.05)
)

plt.title("Percentage Breakdown of Poor/Very Poor Ratings in df_low by Component", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
## Section 2 - Potential SAP (Standard Assessment Procedure) Point Gain

import pandas as pd

# 1. Filter for Houses with low EPC ratings (E, F, or G)
df_houses = df_midlands[
    (df_midlands['PROPERTY_TYPE'] == 'House') & 
    (df_midlands['CURRENT_ENERGY_RATING'].isin(['E', 'F', 'G']))
].copy()

# 2. Glazing setup
modern_glazing_types = [
    'double glazing installed during or after 2002',
    'triple glazing',
    'triple, known data'
]

# 3. Wall insulation setup
wall_u_values = df_houses['WALLS_DESCRIPTION'].str.extract(r'(\d+\.\d+)\s*W/m')[0].astype(float)

good_wall_terms = [
    'filled cavity',
    'insulated',
    'with external insulation',
    'with internal insulation',
    'with additional insulation'
]

# 4. Create Boolean Flags
df_houses['HAS_LOFT_300MM'] = df_houses['ROOF_DESCRIPTION'].str.contains(r'300|400|350|250', case=False, na=False)

df_houses['HAS_CAVITY_INS'] = (
    df_houses['WALLS_DESCRIPTION'].str.contains('|'.join(good_wall_terms), case=False, na=False) |
    (wall_u_values <= 0.35)
)

df_houses['HAS_MODERN_GLAZING'] = df_houses['GLAZED_TYPE'].isin(modern_glazing_types)

# 5. Calculate potential SAP point gain
df_houses['POTENTIAL_GAIN'] = df_houses['POTENTIAL_ENERGY_EFFICIENCY'] - df_houses['CURRENT_ENERGY_EFFICIENCY']

# 6. Filter unimproved houses (missing Loft AND missing Wall AND missing Glazing)
unimproved_filter = (
    (~df_houses['HAS_LOFT_300MM']) & 
    (~df_houses['HAS_CAVITY_INS']) & 
    (~df_houses['HAS_MODERN_GLAZING'])
)

unimproved = df_houses[unimproved_filter]

print(f"Average Potential SAP Point Gain: {unimproved['POTENTIAL_GAIN'].mean():.1f} points")